In [1]:
# === Cell 1: Load and explore dataset files ===
# Rationale: Before any analysis, we need to understand the data structure.
# - metadata.csv → clinical info per sample (patient, group, age, etc.)
# - seqtab_nochim_export.xlsx → sequence abundance table (ASVs) per sample
# - taxa_species_export.xlsx → taxonomic classification of each ASV
# - README.md → dataset description

import pandas as pd

# Clinical metadata
metadata = pd.read_csv('../data/metadata.csv')
print("=== METADATA ===")
print(f"Shape: {metadata.shape[0]} samples x {metadata.shape[1]} columns")
print(f"\nColumns: {list(metadata.columns)}")
print(f"\n{metadata.head()}")

=== METADATA ===
Shape: 59 samples x 1 columns

Columns: ['SampleID;host_disease;SampleName;DiseaseStatus;Sex;Age']

  SampleID;host_disease;SampleName;DiseaseStatus;Sex;Age
0     SRR29923448;CRC1;CT8;Colorectal cancer;male;76    
1   SRR29923449;CRC2;CA1;Colorectal cancer;female;71    
2            SRR29923450;Healthy1;H5;Healthy;male;58    
3  SRR29923451;Polype1;AP20;Adenomatous Polyps;fe...    
4            SRR29923452;Healthy2;H8;Healthy;male;63    


In [ ]:
# === Cell 2: Reload metadata with correct separator ===
# The CSV uses semicolons (;) as delimiter instead of commas.
# We also inspect data types and unique values to understand the clinical groups.

metadata = pd.read_csv('../data/metadata.csv', sep=';')
print("=== METADATA ===")
print(f"Shape: {metadata.shape[0]} samples x {metadata.shape[1]} columns")
print(f"\nColumns: {list(metadata.columns)}")
print(f"\n{metadata.head(10)}")
print(f"\n=== DISEASE GROUPS ===")
print(metadata['DiseaseStatus'].value_counts())
print(f"\n=== SEX DISTRIBUTION ===")
print(metadata['Sex'].value_counts())
print(f"\n=== AGE STATS ===")
print(metadata['Age'].describe())

=== METADATA ===
Shape: 59 samples x 6 columns

Columns: ['SampleID', 'host_disease', 'SampleName', 'DiseaseStatus', 'Sex', 'Age']

      SampleID host_disease SampleName       DiseaseStatus     Sex  Age
0  SRR29923448         CRC1        CT8   Colorectal cancer    male   76
1  SRR29923449         CRC2        CA1   Colorectal cancer  female   71
2  SRR29923450     Healthy1         H5             Healthy    male   58
3  SRR29923451      Polype1       AP20  Adenomatous Polyps  female   69
4  SRR29923452     Healthy2         H8             Healthy    male   63
5  SRR29923453     Healthy3         H9             Healthy    male   76
6  SRR29923454     Healthy4        H10             Healthy  female   79
7  SRR29923455     Healthy5        H11             Healthy  female   79
8  SRR29923456      Polype2      APA13  Adenomatous Polyps    male   54
9  SRR29923457     Healthy6         H7             Healthy    male   58

=== DISEASE GROUPS ===
DiseaseStatus
Colorectal cancer     21
Healthy      

In [ ]:
# === Cell 3: Load ASV abundance table and taxonomy ===
# Rationale: The core of microbiome analysis is the abundance table (samples x ASVs)
# and the taxonomy table (ASV → Kingdom, Phylum, ..., Species).
# We need to understand their dimensions and structure before merging them.

# ASV abundance table: rows = ASVs, columns = samples (usually)
asv_table = pd.read_excel('../data/seqtab_nochim_export.xlsx')
print("=== ASV ABUNDANCE TABLE ===")
print(f"Shape: {asv_table.shape}")
print(f"Columns (first 5): {list(asv_table.columns[:5])}")
print(f"\n{asv_table.iloc[:5, :5]}")

print("\n")

# Taxonomy table: maps each ASV to its taxonomic classification
taxonomy = pd.read_excel('../data/taxa_species_export.xlsx')
print("=== TAXONOMY TABLE ===")
print(f"Shape: {taxonomy.shape}")
print(f"Columns: {list(taxonomy.columns)}")
print(f"\n{taxonomy.head()}")

=== ASV ABUNDANCE TABLE ===
Shape: (59, 6694)
Columns (first 5): ['Unnamed: 0', 'CCTACGGGAGGCAGCAGTGATTAACCTTTAGCAATAAACGAAAGTTTAACTAAGCTATACTAACCCCAGGGTTGGTCAATTTCGTGCCAGCCACCGCGGTCACACGATTAACCCAAGTCAATAGAAGCCGGCGTAAAGAGTGTTTTAGATCACCCCCTCCCCAATAAAGCTAAAACTCACCTGAGTTGTAAAAAACTCCAGTTGACACAAAATAGACTACGAAAGTGGCTTTAACATATCTGAACACACAATAGCTAAGACCCAAACTGGGATTAGATACCCTGGTAGTC', 'CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCCTGATGCAGCGACGCCGCGTGAGCGAAGAAGTATTTCGGTATGTAAAGCTCTATCAGCAGGGAAGAAAATGACGGTACCTGACTAAGAAGCACCGGCTAAATACGTGCCAGCAGCCGCGGTAATACGTATGGTGCAAGCGTTATCCGGATTTACTGGGTGTAAAGGGAGCGTAGACGGATGGGCAAGTCTGATGTGAAAACCCGGGGCTCAACCCCGGGACTGCATTGGAAACTGTTCATCTAGAGTGCTGGAGAGGTAAGTGGAATTCCTAGTGTAGCGGTGAAATGCGTAGATATTAGGAGGAACACCAGTGGCGAAGGCGGCTTACTGGACAGTAACTGACGTTGAGGCTCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTC', 'CCTACGGGAGGCAGCAGTGATTAACCTTTAGCAATAAACGAAAGTTTAACTAAGCTATACTAACCCCAGGGTTGGTCAATTTCGTGCCAGCCACCGCGGTCACACGATTAACCCAAGTCAATAGAAGCCGGCGTAAAGAGTGTTTTAGATCACCCCCTCCCCAATAAAGCTAAAACTCACC

In [4]:
# === Cell 4: Clean column names and verify table alignment ===
# Rationale: Both tables use DNA sequences as identifiers.
# The ASV table has sequences as column names, the taxonomy table has them as rows.
# We need to:
# 1. Rename the 'Unnamed: 0' columns to meaningful names
# 2. Check that the ASV IDs match between both tables
# 3. Set proper indices for easier merging later

# Rename identifier columns
asv_table = asv_table.rename(columns={'Unnamed: 0': 'SampleID'})
taxonomy = taxonomy.rename(columns={'Unnamed: 0': 'ASV_sequence'})

# Set SampleID as index for the abundance table
asv_table = asv_table.set_index('SampleID')

print(f"ASV table: {asv_table.shape[0]} samples x {asv_table.shape[1]} ASVs")
print(f"Taxonomy table: {taxonomy.shape[0]} ASVs x {taxonomy.shape[1]} levels")

# Check if ASV sequences match between tables
asv_in_abundance = set(asv_table.columns)
asv_in_taxonomy = set(taxonomy['ASV_sequence'])
shared = asv_in_abundance & asv_in_taxonomy

print(f"\nASVs in abundance table: {len(asv_in_abundance)}")
print(f"ASVs in taxonomy table:  {len(asv_in_taxonomy)}")
print(f"Shared (matched):        {len(shared)}")
print(f"Only in abundance:       {len(asv_in_abundance - asv_in_taxonomy)}")
print(f"Only in taxonomy:        {len(asv_in_taxonomy - asv_in_abundance)}")

ASV table: 59 samples x 6693 ASVs
Taxonomy table: 6693 ASVs x 8 levels

ASVs in abundance table: 6693
ASVs in taxonomy table:  6693
Shared (matched):        6693
Only in abundance:       0
Only in taxonomy:        0


In [ ]:
# === Cell 5: Display tables in a readable format ===
# Jupyter can render pandas DataFrames as formatted tables
# instead of raw text. We use .head() to show just the first rows,
# and .style for better readability. We also look at the taxonomy
# distribution to understand what types of bacteria dominate.

from IPython.display import display

# Show abundance table (first 5 samples, first 6 ASVs)
print("=== ASV ABUNDANCE TABLE (subset) ===")
display(asv_table.iloc[:5, :6])

print("\n=== TAXONOMY TABLE (first 10 rows) ===")
display(taxonomy.head(10))

# What phyla are most common?
print("\n=== TOP 10 PHYLA ===")
display(taxonomy['Phylum'].value_counts().head(10))

=== ASV ABUNDANCE TABLE (subset) ===


,CCTACGGGAGGCAGCAGTGATTAACCTTTAGCAATAAACGAAAGTTTAACTAAGCTATACTAACCCCAGGGTTGGTCAATTTCGTGCCAGCCACCGCGGTCACACGATTAACCCAAGTCAATAGAAGCCGGCGTAAAGAGTGTTTTAGATCACCCCCTCCCCAATAAAGCTAAAACTCACCTGAGTTGTAAAAAACTCCAGTTGACACAAAATAGACTACGAAAGTGGCTTTAACATATCTGAACACACAATAGCTAAGACCCAAACTGGGATTAGATACCCTGGTAGTC,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCCTGATGCAGCGACGCCGCGTGAGCGAAGAAGTATTTCGGTATGTAAAGCTCTATCAGCAGGGAAGAAAATGACGGTACCTGACTAAGAAGCACCGGCTAAATACGTGCCAGCAGCCGCGGTAATACGTATGGTGCAAGCGTTATCCGGATTTACTGGGTGTAAAGGGAGCGTAGACGGATGGGCAAGTCTGATGTGAAAACCCGGGGCTCAACCCCGGGACTGCATTGGAAACTGTTCATCTAGAGTGCTGGAGAGGTAAGTGGAATTCCTAGTGTAGCGGTGAAATGCGTAGATATTAGGAGGAACACCAGTGGCGAAGGCGGCTTACTGGACAGTAACTGACGTTGAGGCTCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTC,CCTACGGGAGGCAGCAGTGATTAACCTTTAGCAATAAACGAAAGTTTAACTAAGCTATACTAACCCCAGGGTTGGTCAATTTCGTGCCAGCCACCGCGGTCACACGATTAACCCAAGTCAATAGAAGCCGGCGTAAAGAGTGTTTTAGATCACCCCCTCCCCAATAAAGCTAAAACTCACCTGAGTTGTAAAAAACTCCAGTTGACACAAAATAGACTACGAAAGTGGCTTTAACATATCTGAACACACAATAGCTAAGACCCAAACTGGGATTAGATACCCGGGTAGTC,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCCTGATGCAGCAACGCCGCGTGAGTGATGACGGCCTTCGGGTTGTAAAGCTCTGTCTTCAGGGACGATAATGACGGTACCTGAGGAGGAAGCCACGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGTGGCGAGCGTTGTCCGGATTTACTGGGCGTAAAGGGAGCGTAGGCGGACTTTTAAGTGAGATGTGAAATACCCGGGCTCAACTTGGGTGCTGCATTTCAAACTGGAAGTCTAGAGTGCAGGAGAGGAGAATGGAATTCCTAGTGTAGCGGTGAAATGCGTAGAGATTAGGAAGAACACCAGTGGCGAAGGCGATTCTCTGGACTGTAACTGACGCTGAGGCTCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTC,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCCTGATGCAGCGACGCCGCGTGAGCGAAGAAGTATTTCGGTATGTAAAGCTCTATCAGCAGGGAAGAAGAAATGACGGTACCTGACTAAGAAGCACCGGCTAAATACGTGCCAGCAGCCGCGGTAATACGTATGGTGCAAGCGTTATCCGGATTTACTGGGTGTAAAGGGAGCGCAGGCGGAAGGCTAAGTCTGATGTGAAAGCCCGGGGCTCAACCCCGGTACTGCATTGGAAACTGGTCATCTAGAGTGTCGGAGGGGTAAGTGGAATTCCTAGTGTAGCGGTGAAATGCGTAGATATTAGGAGGAACACCAGTGGCGAAGGCGGCTTACTGGACGATAACTGACGCTGAGGCTCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTC,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCCTGATGCAGCGACGCCGCGTGAGCGAAGAAGTATTTCGGTATGTAAAGCTCTATCAGCAGGGAAGAAAATGACGGTACCTGACTAAGAAGCACCGGCTAAATACGTGCCAGCAGCCGCGGTAATACGTATGGTGCAAGCGTTATCCGGATTTACTGGGTGTAAAGGGAGCGTAGACGGATGGGCAAGTCTGATGTGAAAACCCGGGGCTCAACCCCGGGACTGCATTGGAAACTGTTCATCTAGAGTGCTGGAGAGGTAAGTGGAATTCCTAGTGTAGCGGTGAAATGCGTAGATATTAGGAGGAACACCAGTGGCGAAGGCGGCTTACTGGACAGTAACTGACGTTGAGGCTCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCGGGTAGTC
SampleID,,,,,,
CRC1,34,0,20,0,76,0
CRC10,58,25,0,0,0,11
CRC11,176,48,153,0,40,0
CRC12,0,113,73,0,49,71
CRC13,0,0,34,0,69,0



=== TAXONOMY TABLE (first 10 rows) ===


,ASV_sequence,Kingdom,Phylum,Class,Order,Family,Genus,Species
0,CCTACGGGAGGCAGCAGTGATTAACCTTTAGCAATAAACGAAAGTT...,Bacteria,Proteobacteria,Alphaproteobacteria,Rickettsiales,Mitochondria,NaN,NaN
1,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCC...,Bacteria,Firmicutes,Clostridia,Lachnospirales,Lachnospiraceae,[Ruminococcus] torques group,NaN
2,CCTACGGGAGGCAGCAGTGATTAACCTTTAGCAATAAACGAAAGTT...,Bacteria,Proteobacteria,NaN,NaN,NaN,NaN,NaN
3,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCC...,Bacteria,Firmicutes,Clostridia,Clostridiales,Clostridiaceae,Clostridium sensu stricto 1,NaN
4,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCC...,Bacteria,Firmicutes,Clostridia,Lachnospirales,Lachnospiraceae,Roseburia,inulinivorans
5,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCC...,Bacteria,Firmicutes,Clostridia,Lachnospirales,Lachnospiraceae,[Ruminococcus] torques group,NaN
6,CCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCC...,Bacteria,Firmicutes,Clostridia,Lachnospirales,Lachnospiraceae,[Ruminococcus] torques group,NaN
7,CCTACGGGAGGCAGCAGTGATTAACCTTTAGCAATAAACGAAAGTT...,Bacteria,Proteobacteria,Alphaproteobacteria,Rickettsiales,Mitochondria,NaN,NaN
8,CCTACGGGAGGCAGCAGTGATTAACCTTTAGCAATAAACGAAAGTT...,Bacteria,Proteobacteria,Alphaproteobacteria,Rickettsiales,Mitochondria,NaN,NaN
9,CCTACGGGAGGCTGCAGTGATTAACCTTTAGCAATAAACGAAAGTT...,Bacteria,Proteobacteria,Alphaproteobacteria,Rickettsiales,Mitochondria,NaN,NaN



=== TOP 10 PHYLA ===


Phylum
Firmicutes           4466
Proteobacteria        714
Fusobacteriota        398
Bacteroidota          329
Actinobacteriota      312
Campylobacterota       65
Verrucomicrobiota      54
Cyanobacteria          35
Patescibacteria        18
Desulfobacterota       16
Name: count, dtype: int64

In [ ]:
# === Cell 6: Display abundance table with truncated ASV names ===
# Rationale: The full DNA sequences are too long to display as column headers.
# This is only for visualization — the original data stays intact.


# Show abundance table (first 5 samples, first 6 ASVs)
asv_display = asv_table.iloc[:5, :6].copy()
# We truncate them to the first 20 characters + "..." for readability.
asv_display.columns = [seq[:20] + '...' for seq in asv_display.columns]
display(asv_display)

,CCTACGGGAGGCAGCAGTGA...,CCTACGGGAGGCAGCAGTGG...,CCTACGGGAGGCAGCAGTGA...,CCTACGGGAGGCAGCAGTGG...,CCTACGGGAGGCAGCAGTGG...,CCTACGGGAGGCAGCAGTGG...
SampleID,,,,,,
CRC1,34,0,20,0,76,0
CRC10,58,25,0,0,0,11
CRC11,176,48,153,0,40,0
CRC12,0,113,73,0,49,71
CRC13,0,0,34,0,69,0
